# Análise exploratória: recursos partidários e estrutura de listas (2018 e 2022)

Notebook de exploração descritiva do dataset `rrd_df_novo.parquet`, restrito a candidatos a Deputado Federal nas eleições ordinárias de 2018 e 2022 (1.026 eleitos + 16.279 não eleitos).

---

## 1. Recursos partidários entre eleitos e não eleitos

**Pergunta:** Entre os eleitos, qual o share médio intralista? Quantos ficaram sem recursos de partido? A maioria depende de fontes partidárias ou não partidárias?

**Resultados (eleitos, 2018+2022, n=1.026):**
- Share médio intralista (`prop_vr_receita_candidato`): **27,4%** — eleitos concentram mais de 1/4 dos recursos FEFC/FP da lista
- Apenas **3,9%** dos eleitos não receberam nenhum recurso de partido
- **83,8%** receberam mais de partido do que de outras fontes

**Comparação por ano:**

| | 2018 | 2022 |
|---|---|---|
| Share médio intralista | 37,6% | 17,2% |
| Sem recursos de partido | 6,2% | 1,6% |
| Partido > outros | 77,2% | 90,4% |
| Outros > partido | 22,8% | 9,6% |

Em 2022 os recursos chegam a *mais* candidatos eleitos (queda de sem-partido), mas o share médio cai pela metade — distribuição mais pulverizada, consistente com a atenuação da coordenação documentada nos modelos.

**Comparação eleitos vs. não eleitos (2018+2022):**
- Share intralista médio: **27,4%** (eleitos) vs. 7,1% (não eleitos) — quase 4×
- Sem recursos de partido: 3,9% (eleitos) vs. 19,0% (não eleitos) — 5× mais frequente entre perdedores

---

## 2. Bancadas solo e candidatos solo: distinção conceitual

**Pergunta:** Quantos eleitos são o único representante do partido no estado? Isso é o mesmo que o partido ter lançado apenas 1 candidato?

**Distinção crucial:**
- **Lista solo** (partido lançou 1 único candidato no estado): caso trivial, sem problema de coordenação
- **Bancada solo** (partido elegeu 1, mas havia 2+ candidatos): caso analiticamente relevante — coordenação ocorreu

**Resultados (eleitos, 2018+2022):**

| | Bancada 2+ | Bancada solo |
|---|---|---|
| Lista solo (trivial) | 0 | 35 (todos em 2018) |
| Lista 2+ cands | 740 | 251 |

- Em 2022 não existe nenhuma lista solo entre os eleitos — reflexo das fusões partidárias
- Os **251 casos de bancada solo com lista competitiva** são o universo analiticamente central

**Recursos por grupo (listas 2+ cands):**
- Bancada solo: share médio 51%, mediana R$977k
- Bancada 2+: share médio 16%, mediana R$1,2M

O candidato solo captura share alto mas recebe *menos* em valor absoluto — o share reflete ausência de competidores internos, não necessariamente maior alocação estratégica.

---

## 3. Bancada solo × magnitude do distrito

**Pergunta:** A magnitude do distrito explica a concentração de bancadas solo?

**Taxa de bancada solo entre eleitos de listas competitivas:**

| Magnitude | 2018 | 2022 |
|---|---|---|
| Pequeno (8–12) | **75,7%** | 16,3% |
| Médio (16–31) | 35,3% | 24,4% |
| Grande (39–70) | 13,9% | 10,1% |

Em 2018, 3 em cada 4 eleitos de distritos pequenos eram o único do partido no estado — bancada solo é quase mecânica em distritos pequenos. A queda abrupta em 2022 reflete consolidação partidária.

**Recursos absolutos (bancada solo vs. 2+):**
- O gap de recursos absolutos é maior em distritos pequenos (R$900k vs. R$1,465k) e desaparece em grandes (R$1,025k vs. R$1,102k)
- O diferencial de *share* persiste em todos os tamanhos, mas é artefato de lista pequena em partidos menos capitalizados

---

## 4. Bancada solo × magnitude × tipo de partido (competitivo / menos competitivo)

**Partidos competitivos (≥20 cadeiras nacionais):** taxa de bancada solo concentrada em distritos pequenos em 2018, colapsa em 2022 (73% → 13%) com as fusões.

**Partidos menos competitivos:** taxa de bancada solo alta e *estável* em médio e grande (68% → 66%; 41% → 34%) — para esses partidos, bancada solo é estrutural, não conjuntural.

**Implicação:** A bancada solo em distritos pequenos e/ou em partidos menos competitivos é predominantemente mecânica. O caso analiticamente mais limpo para coordenação são partidos competitivos em distritos médios e grandes.

---

## 5. Análise por partido selecionado × magnitude × grupo

Partidos analisados: **PL, PSL, PT, UNIÃO, PP, MDB, PSD**

**Destaques:**

- **PSL (2018):** caso extremo de não-coordenação — 58–100% dos eleitos em médio e grande receberam zero de recurso partidário; eleições amplamente autofinanciadas (efeito Bolsonaro)
- **PL (2022):** zero sem-partido em todas as magnitudes; share muito baixo em grandes distritos (5%), medianas baixas (R$542k) — distribuição muito pulverizada por bancada enorme (99 eleitos)
- **UNIÃO (2022):** partido mais capitalizado — medianas acima de R$2,3M em todos os tamanhos, zero sem-partido; bancadas solo em pequeno/médio recebem mais que a 2+, possível sinal de concentração real
- **PT:** zero sem-partido consistente; gap de share solo/2+ estável entre anos; mediana cresce com magnitude para a bancada 2+, consistente com mais recursos alocados a listas maiores
- **PP e MDB:** perfil similar ao PT, sem casos expressivos de sem-partido
- **PSD:** share alto nos solos de pequeno (55%) mas mediana baixa (R$840k); bancada 2+ de pequeno recebe R$1,8M — padrão de bancada solo em lista pobre

## Análise de recursos partidários entre candidatos eleitos (2018 e 2022)

In [ ]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)


In [ ]:
import sys
import numpy as np
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path("figs/src").resolve()))
from cap3_cs_features import gerar_features

df = pd.read_parquet("data/processed/rrd_df_novo.parquet")

# Filtra 2018 e 2022 e computa features (incluindo prop_vr_receita_candidato)
rrd = df[df.ano_eleicao.isin([2018, 2022])].copy()
rrd = gerar_features(rrd)

# Trata NaN em vr_receita_outros como 0 (candidato não recebeu de outras fontes)
rrd["vr_receita_outros_fill"] = rrd["vr_receita_outros"].fillna(0)

eleitos = rrd[rrd.eleito == 1].copy()
print(f"Total de eleitos (2018+2022): {len(eleitos)}")
print(f"  2018: {(eleitos.ano_eleicao == 2018).sum()}  |  2022: {(eleitos.ano_eleicao == 2022).sum()}")

Total de eleitos (2018+2022): 1026
  2018: 513  |  2022: 513


In [3]:
def resumo_eleitos(df_el, label=""):
    n = len(df_el)

    # Q1: Share médio intralista
    share_medio = df_el["prop_vr_receita_candidato"].mean()

    # Q2: Sem nenhum recurso de partido
    sem_partido = (df_el["vr_receita_recursos_partidos"] == 0).sum()

    # Q3: Recursos de partido > outros
    mais_partido = (df_el["vr_receita_recursos_partidos"] > df_el["vr_receita_outros_fill"]).sum()

    # Q4: Maioria de recursos não partidários (outros > partido, excluindo quem tem zero de ambos)
    mais_outros = (df_el["vr_receita_outros_fill"] > df_el["vr_receita_recursos_partidos"]).sum()

    # Empate (ambos iguais e > 0): residual
    empate = n - sem_partido - mais_partido - mais_outros

    print(f"{'─'*55}")
    print(f"  {label}  (n = {n})")
    print(f"{'─'*55}")
    print(f"  Share médio intralista (prop_vr_receita):  {share_medio:.3f}  ({share_medio*100:.1f}%)")
    print(f"  Sem recursos de partido:                   {sem_partido:>4}  ({sem_partido/n*100:.1f}%)")
    print(f"  Partido > outros:                          {mais_partido:>4}  ({mais_partido/n*100:.1f}%)")
    print(f"  Outros > partido:                          {mais_outros:>4}  ({mais_outros/n*100:.1f}%)")
    if empate > 0:
        print(f"  Empate (ambos iguais, > 0):                {empate:>4}  ({empate/n*100:.1f}%)")
    print()


resumo_eleitos(eleitos, "TOTAL 2018+2022")
resumo_eleitos(eleitos[eleitos.ano_eleicao == 2018], "2018")
resumo_eleitos(eleitos[eleitos.ano_eleicao == 2022], "2022")

───────────────────────────────────────────────────────
  TOTAL 2018+2022  (n = 1026)
───────────────────────────────────────────────────────
  Share médio intralista (prop_vr_receita):  0.274  (27.4%)
  Sem recursos de partido:                     40  (3.9%)
  Partido > outros:                           860  (83.8%)
  Outros > partido:                           166  (16.2%)

───────────────────────────────────────────────────────
  2018  (n = 513)
───────────────────────────────────────────────────────
  Share médio intralista (prop_vr_receita):  0.376  (37.6%)
  Sem recursos de partido:                     32  (6.2%)
  Partido > outros:                           396  (77.2%)
  Outros > partido:                           117  (22.8%)

───────────────────────────────────────────────────────
  2022  (n = 513)
───────────────────────────────────────────────────────
  Share médio intralista (prop_vr_receita):  0.172  (17.2%)
  Sem recursos de partido:                      8  (1.6%)
  Part

## Mesma análise para não eleitos

In [4]:
nao_eleitos = rrd[rrd.eleito == 0].copy()
print(f"Total de não eleitos (2018+2022): {len(nao_eleitos)}")
print(f"  2018: {(nao_eleitos.ano_eleicao == 2018).sum()}  |  2022: {(nao_eleitos.ano_eleicao == 2022).sum()}")
print()
resumo_eleitos(nao_eleitos, "NÃO ELEITOS — TOTAL 2018+2022")
resumo_eleitos(nao_eleitos[nao_eleitos.ano_eleicao == 2018], "NÃO ELEITOS — 2018")
resumo_eleitos(nao_eleitos[nao_eleitos.ano_eleicao == 2022], "NÃO ELEITOS — 2022")

Total de não eleitos (2018+2022): 16279
  2018: 7117  |  2022: 9162

───────────────────────────────────────────────────────
  NÃO ELEITOS — TOTAL 2018+2022  (n = 16279)
───────────────────────────────────────────────────────
  Share médio intralista (prop_vr_receita):  0.071  (7.1%)
  Sem recursos de partido:                   3092  (19.0%)
  Partido > outros:                          11445  (70.3%)
  Outros > partido:                          3564  (21.9%)

───────────────────────────────────────────────────────
  NÃO ELEITOS — 2018  (n = 7117)
───────────────────────────────────────────────────────
  Share médio intralista (prop_vr_receita):  0.083  (8.3%)
  Sem recursos de partido:                   2014  (28.3%)
  Partido > outros:                          3848  (54.1%)
  Outros > partido:                          2578  (36.2%)

───────────────────────────────────────────────────────
  NÃO ELEITOS — 2022  (n = 9162)
───────────────────────────────────────────────────────
  Share m

## Bancadas partidárias estaduais de uma pessoa

In [5]:
lista = ["ano_eleicao", "sg_uf", "sg_partido"]

# Eleitos por lista
eleitos_por_lista = (
    rrd.groupby(lista)["eleito"]
    .sum()
    .reset_index()
    .rename(columns={"eleito": "n_eleitos_lista"})
)
rrd2 = rrd.merge(eleitos_por_lista, on=lista, how="left")

# Flag bancada solo
rrd2["bancada_solo"] = rrd2["n_eleitos_lista"] == 1

# Quantos eleitos estão em bancadas solo?
eleitos2 = rrd2[rrd2.eleito == 1].copy()

solo = eleitos2[eleitos2.bancada_solo]
nao_solo = eleitos2[~eleitos2.bancada_solo]

print(f"{'─'*60}")
print(f"  Eleitos em bancadas solo (partido-UF com 1 eleito)")
print(f"{'─'*60}")
print(f"  Total:  {len(solo)}  ({len(solo)/len(eleitos2)*100:.1f}% dos eleitos)")
print(f"  2018:   {(solo.ano_eleicao==2018).sum()}")
print(f"  2022:   {(solo.ano_eleicao==2022).sum()}")
print()

# Listas (partido-UF) com exatamente 1 eleito
n_listas_solo = rrd2[rrd2.bancada_solo].groupby(lista).ngroups
print(f"  Listas partido-UF com bancada solo: {n_listas_solo}")
print(f"  2018: {rrd2[(rrd2.bancada_solo) & (rrd2.ano_eleicao==2018)].groupby(lista).ngroups}")
print(f"  2022: {rrd2[(rrd2.bancada_solo) & (rrd2.ano_eleicao==2022)].groupby(lista).ngroups}")

────────────────────────────────────────────────────────────
  Eleitos em bancadas solo (partido-UF com 1 eleito)
────────────────────────────────────────────────────────────
  Total:  286  (27.9% dos eleitos)
  2018:   201
  2022:   85

  Listas partido-UF com bancada solo: 286
  2018: 201
  2022: 85


In [6]:
cols_rec = ["vr_receita_recursos_partidos", "vr_receita_outros_fill", "prop_vr_receita_candidato"]

def stats_recursos(df_el, label):
    rec_part = df_el["vr_receita_recursos_partidos"]
    outros   = df_el["vr_receita_outros_fill"]
    share    = df_el["prop_vr_receita_candidato"]
    n = len(df_el)
    print(f"  {label}  (n={n})")
    print(f"    Share intralista médio:          {share.mean():.3f}  ({share.mean()*100:.1f}%)")
    print(f"    Recursos partido — mediana:      R$ {rec_part.median():>12,.0f}")
    print(f"    Recursos partido — média:        R$ {rec_part.mean():>12,.0f}")
    print(f"    Sem recursos de partido:         {(rec_part==0).sum():>4}  ({(rec_part==0).mean()*100:.1f}%)")
    print(f"    Partido > outros:                {(rec_part > outros).sum():>4}  ({(rec_part > outros).mean()*100:.1f}%)")
    print()

print(f"{'─'*60}")
print(f"  TOTAL 2018+2022")
print(f"{'─'*60}")
stats_recursos(solo,     "Bancada SOLO  (n_eleitos_lista = 1)")
stats_recursos(nao_solo, "Bancada c/ 2+ eleitos")

for ano in [2018, 2022]:
    print(f"{'─'*60}")
    print(f"  {ano}")
    print(f"{'─'*60}")
    stats_recursos(solo[solo.ano_eleicao==ano],     f"Solo  {ano}")
    stats_recursos(nao_solo[nao_solo.ano_eleicao==ano], f"2+    {ano}")

────────────────────────────────────────────────────────────
  TOTAL 2018+2022
────────────────────────────────────────────────────────────
  Bancada SOLO  (n_eleitos_lista = 1)  (n=286)
    Share intralista médio:          0.570  (57.0%)
    Recursos partido — mediana:      R$      968,130
    Recursos partido — média:        R$    1,118,937
    Sem recursos de partido:            8  (2.8%)
    Partido > outros:                 242  (84.6%)

  Bancada c/ 2+ eleitos  (n=740)
    Share intralista médio:          0.160  (16.0%)
    Recursos partido — mediana:      R$    1,201,600
    Recursos partido — média:        R$    1,294,176
    Sem recursos de partido:           32  (4.3%)
    Partido > outros:                 618  (83.5%)

────────────────────────────────────────────────────────────
  2018
────────────────────────────────────────────────────────────
  Solo  2018  (n=201)
    Share intralista médio:          0.669  (66.9%)
    Recursos partido — mediana:      R$      850,000
    

In [7]:
# Candidatos totais por lista (partido × UF × ano)
cands_por_lista = (
    rrd2.groupby(lista)["nr_candidato"]
    .count()
    .reset_index()
    .rename(columns={"nr_candidato": "n_cands_lista"})
)
rrd3 = rrd2.merge(cands_por_lista, on=lista, how="left")

rrd3["lista_solo"]   = rrd3["n_cands_lista"] == 1   # partido lançou 1 único candidato
rrd3["bancada_solo"] = rrd3["n_eleitos_lista"] == 1  # partido elegeu só 1

eleitos3 = rrd3[rrd3.eleito == 1].copy()

# Crosstab: candidatos eleitos por tipo de lista
tab = pd.crosstab(
    eleitos3["lista_solo"].map({True: "1 candidato na lista", False: "2+ candidatos na lista"}),
    eleitos3["bancada_solo"].map({True: "Bancada solo (1 eleito)", False: "Bancada 2+ eleitos"}),
    margins=True
)
print("Eleitos 2018+2022 — cruzamento lista × bancada\n")
print(tab)
print()

# Por ano
for ano in [2018, 2022]:
    sub = eleitos3[eleitos3.ano_eleicao == ano]
    tab_ano = pd.crosstab(
        sub["lista_solo"].map({True: "1 cand na lista", False: "2+ cands na lista"}),
        sub["bancada_solo"].map({True: "Bancada solo", False: "Bancada 2+"}),
        margins=True
    )
    print(f"{ano}\n{tab_ano}\n")

Eleitos 2018+2022 — cruzamento lista × bancada

bancada_solo            Bancada 2+ eleitos  Bancada solo (1 eleito)   All
lista_solo                                                               
1 candidato na lista                     0                       35    35
2+ candidatos na lista                 740                      251   991
All                                    740                      286  1026

2018
bancada_solo       Bancada 2+  Bancada solo  All
lista_solo                                      
1 cand na lista             0            35   35
2+ cands na lista         312           166  478
All                       312           201  513

2022
bancada_solo       Bancada 2+  Bancada solo  All
lista_solo                                      
2+ cands na lista         428            85  513
All                       428            85  513



In [12]:

g_lista_solo    = eleitos3[eleitos3.lista_solo]
g_bancada_solo  = eleitos3[~eleitos3.lista_solo &  eleitos3.bancada_solo]
g_bancada_multi = eleitos3[~eleitos3.lista_solo & ~eleitos3.bancada_solo]

grupos = [
    (g_lista_solo,    "Lista solo (1 cand, 1 eleito) — trivial"),
    (g_bancada_solo,  "Bancada solo (2+ cands, 1 eleito)"),
    (g_bancada_multi, "Bancada 2+   (2+ cands, 2+ eleitos)"),
]

print(f"{'─'*65}")
print(f"  Recursos partidários por grupo — 2018+2022")
print(f"{'─'*65}\n")
for df_g, label in grupos:
    rec  = df_g["vr_receita_recursos_partidos"]
    sh   = df_g["prop_vr_receita_candidato"]
    out  = df_g["vr_receita_outros_fill"]
    print(f"  {label}  (n={len(df_g)})")
    print(f"    Share intralista médio:     {sh.mean():.3f}  ({sh.mean()*100:.1f}%)")
    print(f"    Recursos partido mediana:   R$ {rec.median():>12,.0f}")
    print(f"    Recursos partido média:     R$ {rec.mean():>12,.0f}")
    print(f"    Sem recursos de partido:    {(rec==0).sum():>3}  ({(rec==0).mean()*100:.1f}%)")
    print(f"    Partido > outros:           {(rec>out).sum():>3}  ({(rec>out).mean()*100:.1f}%)")
    print()

print(f"\n{'─'*65}")
print(f"  Mesmo detalhamento por ano")
print(f"{'─'*65}")
for ano in [2018, 2022]:
    print(f"\n  {ano}")
    for df_g, label in grupos:
        sub = df_g[df_g.ano_eleicao == ano]
        if len(sub) == 0:
            continue
        rec = sub["vr_receita_recursos_partidos"]
        sh  = sub["prop_vr_receita_candidato"]
        print(f"    {label}  (n={len(sub)})  |  share={sh.mean():.3f}  mediana=R${rec.median():,.0f}  sem_partido={(rec==0).sum()} ({(rec==0).mean()*100:.1f}%)")


─────────────────────────────────────────────────────────────────
  Recursos partidários por grupo — 2018+2022
─────────────────────────────────────────────────────────────────

  Lista solo (1 cand, 1 eleito) — trivial  (n=35)
    Share intralista médio:     1.000  (100.0%)
    Recursos partido mediana:   R$      896,500
    Recursos partido média:     R$    1,065,340
    Sem recursos de partido:      0  (0.0%)
    Partido > outros:            31  (88.6%)

  Bancada solo (2+ cands, 1 eleito)  (n=251)
    Share intralista médio:     0.510  (51.0%)
    Recursos partido mediana:   R$      976,720
    Recursos partido média:     R$    1,126,410
    Sem recursos de partido:      8  (3.2%)
    Partido > outros:           211  (84.1%)

  Bancada 2+   (2+ cands, 2+ eleitos)  (n=740)
    Share intralista médio:     0.160  (16.0%)
    Recursos partido mediana:   R$    1,201,600
    Recursos partido média:     R$    1,294,176
    Sem recursos de partido:     32  (4.3%)
    Partido > outros:     

In [13]:

assert "dm_cat" in eleitos3.columns

listas_competitivas = eleitos3[~eleitos3.lista_solo].copy()

taxa_solo = (
    listas_competitivas
    .groupby(["dm_cat", "ano_eleicao"])["bancada_solo"]
    .agg(n_eleitos="count", n_solo="sum")
    .assign(pct_solo=lambda x: x["n_solo"] / x["n_eleitos"] * 100)
    .reset_index()
)
print("Taxa de bancada solo entre eleitos (listas 2+ cands) por magnitude\n")
print(taxa_solo.to_string(index=False))
print()

print(f"{'─'*72}")
print(f"  Share intralista médio e mediana de recursos por magnitude × grupo")
print(f"{'─'*72}\n")

for dm in listas_competitivas["dm_cat"].cat.categories:
    sub = listas_competitivas[listas_competitivas["dm_cat"] == dm]
    s_solo  = sub[sub.bancada_solo]
    s_multi = sub[~sub.bancada_solo]
    print(f"  Magnitude: {dm}  (n eleitos={len(sub)})")
    for grp, label in [(s_solo, "bancada solo"), (s_multi, "bancada 2+")]:
        if len(grp) == 0:
            continue
        rec = grp["vr_receita_recursos_partidos"]
        sh  = grp["prop_vr_receita_candidato"]
        print(f"    {label:20s}  n={len(grp):>3}  share={sh.mean():.3f}  "
              f"mediana=R${rec.median():>10,.0f}  média=R${rec.mean():>10,.0f}")
    print()


Taxa de bancada solo entre eleitos (listas 2+ cands) por magnitude

        dm_cat  ano_eleicao  n_eleitos  n_solo  pct_solo
Pequeno (8–12)         2018        103      78 75.728155
Pequeno (8–12)         2022        129      21 16.279070
 Médio (16–31)         2018        167      59 35.329341
 Médio (16–31)         2022        176      43 24.431818
Grande (39–70)         2018        208      29 13.942308
Grande (39–70)         2022        208      21 10.096154

────────────────────────────────────────────────────────────────────────
  Share intralista médio e mediana de recursos por magnitude × grupo
────────────────────────────────────────────────────────────────────────

  Magnitude: Pequeno (8–12)  (n eleitos=232)
    bancada solo          n= 99  share=0.573  mediana=R$   900,000  média=R$ 1,031,820
    bancada 2+            n=133  share=0.228  mediana=R$ 1,464,742  média=R$ 1,440,556

  Magnitude: Médio (16–31)  (n eleitos=343)
    bancada solo          n=102  share=0.514  median

C:\Users\yuri.taba\AppData\Local\Temp\ipykernel_23184\3085132281.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["dm_cat", "ano_eleicao"])["bancada_solo"]


In [14]:

assert "tipo_partido" in eleitos3.columns

lc = listas_competitivas

print("Taxa de bancada solo por magnitude × tipo_partido × ano\n")
taxa2 = (
    lc.groupby(["dm_cat", "tipo_partido", "ano_eleicao"], observed=True)["bancada_solo"]
    .agg(n="count", n_solo="sum")
    .assign(pct_solo=lambda x: x["n_solo"] / x["n"] * 100)
    .reset_index()
)
print(taxa2.to_string(index=False))
print()

print(f"{'─'*80}")
print(f"  Share intralista e recursos: magnitude × tipo_partido × grupo (2018+2022)")
print(f"{'─'*80}\n")

for dm in lc["dm_cat"].cat.categories:
    for tp in ["Competitivo", "Menos competitivo"]:
        sub = lc[(lc["dm_cat"] == dm) & (lc["tipo_partido"] == tp)]
        if len(sub) == 0:
            continue
        s_solo  = sub[sub.bancada_solo]
        s_multi = sub[~sub.bancada_solo]
        print(f"  {dm} | {tp}  (n={len(sub)})")
        for grp, label in [(s_solo, "bancada solo"), (s_multi, "bancada 2+")]:
            if len(grp) == 0:
                print(f"    {'bancada solo' if label=='bancada solo' else 'bancada 2+':20s}  n=  0")
                continue
            rec = grp["vr_receita_recursos_partidos"]
            sh  = grp["prop_vr_receita_candidato"]
            sem = (rec == 0).mean() * 100
            print(f"    {label:20s}  n={len(grp):>3}  share={sh.mean():.3f}  "
                  f"mediana=R${rec.median():>10,.0f}  sem_partido={sem:.1f}%")
        print()


Taxa de bancada solo por magnitude × tipo_partido × ano

        dm_cat      tipo_partido  ano_eleicao   n  n_solo  pct_solo
Pequeno (8–12)       Competitivo         2018  86      63 73.255814
Pequeno (8–12)       Competitivo         2022 113      15 13.274336
Pequeno (8–12) Menos competitivo         2018  17      15 88.235294
Pequeno (8–12) Menos competitivo         2022  16       6 37.500000
 Médio (16–31)       Competitivo         2018 130      34 26.153846
 Médio (16–31)       Competitivo         2022 135      16 11.851852
 Médio (16–31) Menos competitivo         2018  37      25 67.567568
 Médio (16–31) Menos competitivo         2022  41      27 65.853659
Grande (39–70)       Competitivo         2018 154       7  4.545455
Grande (39–70)       Competitivo         2022 150       1  0.666667
Grande (39–70) Menos competitivo         2018  54      22 40.740741
Grande (39–70) Menos competitivo         2022  58      20 34.482759

──────────────────────────────────────────────────────────

In [15]:

df.query("ano_eleicao == 2022").groupby("sg_partido").eleito.sum().sort_values(ascending=False).head(20)

sg_partido
PL               99
PT               69
UNIÃO            59
PP               47
MDB              42
PSD              42
REPUBLICANOS     40
PDT              17
PSB              14
PSDB             13
PODE             12
PSOL             12
AVANTE            7
PV                6
PC do B           6
PSC               6
CIDADANIA         5
SOLIDARIEDADE     4
PATRIOTA          4
NOVO              3
Name: eleito, dtype: int32

## Análise por partido selecionado × magnitude × grupo (bancada solo / 2+)

In [16]:

partidos_foco = ["PL", "PSL", "PT", "UNIÃO", "PP", "MDB", "PSD"]

lc_foco = lc[lc["sg_partido"].isin(partidos_foco)].copy()

print("Taxa de bancada solo por partido × magnitude × ano (listas 2+ cands)\n")
taxa_p = (
    lc_foco
    .groupby(["sg_partido", "dm_cat", "ano_eleicao"], observed=True)["bancada_solo"]
    .agg(n="count", n_solo="sum")
    .assign(pct_solo=lambda x: x["n_solo"] / x["n"] * 100)
    .reset_index()
)
print(taxa_p.to_string(index=False))

print(f"\n{'─'*82}")
print(f"  Share intralista e recursos por partido × magnitude × grupo (2018+2022)")
print(f"{'─'*82}\n")

for partido in partidos_foco:
    sub_p = lc_foco[lc_foco["sg_partido"] == partido]
    if len(sub_p) == 0:
        continue
    anos = sorted(sub_p["ano_eleicao"].unique())
    print(f"  ══ {partido}  (anos disponíveis: {anos}, n eleitos={len(sub_p)}) ══")
    for dm in lc["dm_cat"].cat.categories:
        sub = sub_p[sub_p["dm_cat"] == dm]
        if len(sub) == 0:
            continue
        s_solo  = sub[sub.bancada_solo]
        s_multi = sub[~sub.bancada_solo]
        print(f"    {dm}  (n={len(sub)})")
        for grp, label in [(s_solo, "bancada solo"), (s_multi, "bancada 2+")]:
            if len(grp) == 0:
                print(f"      {label:20s}  n=  0")
                continue
            rec = grp["vr_receita_recursos_partidos"]
            sh  = grp["prop_vr_receita_candidato"]
            sem = (rec == 0).mean() * 100
            print(f"      {label:20s}  n={len(grp):>3}  share={sh.mean():.3f}  "
                  f"mediana=R${rec.median():>10,.0f}  sem_partido={sem:.1f}%")
    print()


Taxa de bancada solo por partido × magnitude × ano (listas 2+ cands)

sg_partido         dm_cat  ano_eleicao  n  n_solo   pct_solo
       MDB Pequeno (8–12)         2018  9       5  55.555556
       MDB Pequeno (8–12)         2022 11       1   9.090909
       MDB  Médio (16–31)         2018 15       2  13.333333
       MDB  Médio (16–31)         2022 21       4  19.047619
       MDB Grande (39–70)         2018  9       0   0.000000
       MDB Grande (39–70)         2022 10       1  10.000000
        PL Pequeno (8–12)         2022 24       3  12.500000
        PL  Médio (16–31)         2022 33       0   0.000000
        PL Grande (39–70)         2022 42       0   0.000000
        PP Pequeno (8–12)         2018 10       8  80.000000
        PP Pequeno (8–12)         2022 17       2  11.764706
        PP  Médio (16–31)         2018 13       3  23.076923
        PP  Médio (16–31)         2022 16       1   6.250000
        PP Grande (39–70)         2018 12       0   0.000000
        PP Gran